# 19A2 — Temporal AIA CNN–GRU Development Baseline

## Purpose
Train the first serious temporal AIA-only model using the frozen Cycle-24 development protocol and the fully staged local AIA cache.

Supervisor-methodology alignment:
**six-channel AIA frame → shared CNN spatial encoder → three-frame GRU temporal encoder → 48 h M/X probability**

This notebook does **not** use Cycle-25 data and does **not** fit the final calibrator or select the final operating threshold.

## Locked scientific contract
- Forecast target: same-active-region M/X flare occurrence in `(t, t+48 h]`
- AIA channels: 94, 131, 171, 193, 211, 335 Å
- Stored labels: `aia94, aia131, aia171, aia193, aia211, aia335`
- Temporal history: t−288, t−192, t−96 min
- Source tensor: `(512, 512, 6)` float32
- Authoritative label: manifest `label_48h_final`
- Embedded NPZ `y` ignored
- Cycle-25 untouched
- Calibration holdout untouched
- Threshold holdout untouched

## Development strategy
1. Use only `cycle24_final_refit_pool`.
2. Build a deterministic region-grouped internal validation split.
3. Fit preprocessing statistics on internal-training data only.
4. Select epoch by validation PR-AUC.
5. Refit the fixed architecture from scratch on the complete `cycle24_final_refit_pool` for the selected number of epochs.
6. Save the frozen base model for 19A3 calibration/threshold freezing.


In [ ]:
from pathlib import Path
import json, math, random, time, sys
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)

SEED = 20260917
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

HOME = Path.home()
CACHE_ROOT = Path("/mnt/disks/aia-cache/cycle24")
PREP = HOME / "aia19_cycle24_cache_prep"
TARGET_MAP = PREP / "cycle24_temporal_targets_local_paths.csv.gz"

OUT = HOME / "aia19_cnn_gru_20260917"
(OUT / "models").mkdir(parents=True, exist_ok=True)
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "logs").mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPECTED_CHANNELS = ["aia94","aia131","aia171","aia193","aia211","aia335"]
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
MAX_EPOCHS = 20
PATIENCE = 4
LR = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.30

print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

assert TARGET_MAP.exists(), TARGET_MAP
assert CACHE_ROOT.exists(), CACHE_ROOT


## 1. Load the staged target map and isolate the model-development pool

In [ ]:
rows = pd.read_csv(TARGET_MAP)

required = [
    "target_sample_id",
    "label_48h_final",
    "role",
    "region_component_id",
    "local_tminus288",
    "local_tminus192",
    "local_tminus96",
]
missing = [c for c in required if c not in rows.columns]
if missing:
    raise RuntimeError(f"Missing target-map columns: {missing}")

assert len(rows) == 64725
assert int(rows["label_48h_final"].sum()) == 2094

fit_pool = rows[rows["role"].eq("cycle24_final_refit_pool")].copy()
cal_pool = rows[rows["role"].eq("cycle24_calibration_holdout")].copy()
thr_pool = rows[rows["role"].eq("cycle24_threshold_holdout")].copy()

print("Final refit pool:", len(fit_pool), "positives:", int(fit_pool["label_48h_final"].sum()))
print("Calibration holdout:", len(cal_pool), "positives:", int(cal_pool["label_48h_final"].sum()))
print("Threshold holdout:", len(thr_pool), "positives:", int(thr_pool["label_48h_final"].sum()))

assert len(fit_pool) > 0
assert len(cal_pool) > 0
assert len(thr_pool) > 0

assert set(fit_pool["target_sample_id"]).isdisjoint(set(cal_pool["target_sample_id"]))
assert set(fit_pool["target_sample_id"]).isdisjoint(set(thr_pool["target_sample_id"]))


## 2. Deterministic grouped internal validation split

The split is by `region_component_id`, preventing active-region component overlap.
The latest 15% of region components, ordered by their latest target timestamp, are used for internal validation.
This internal split is used only for epoch selection.


In [ ]:
def sid_to_timestamp(s):
    return pd.to_datetime(str(s)[:13], format="%Y%m%d_%H%M", utc=True)

fit_pool["target_time"] = fit_pool["target_sample_id"].map(sid_to_timestamp)

groups = (
    fit_pool.groupby("region_component_id")
    .agg(
        n=("target_sample_id","size"),
        positives=("label_48h_final","sum"),
        last_time=("target_time","max"),
    )
    .sort_values(["last_time","region_component_id"])
)

n_val_groups = max(1, math.ceil(len(groups) * 0.15))
val_groups = set(groups.tail(n_val_groups).index)
train_groups = set(groups.index) - val_groups

internal_train = fit_pool[fit_pool["region_component_id"].isin(train_groups)].copy()
internal_val = fit_pool[fit_pool["region_component_id"].isin(val_groups)].copy()

assert set(internal_train["region_component_id"]).isdisjoint(set(internal_val["region_component_id"]))
assert set(internal_train["target_sample_id"]).isdisjoint(set(internal_val["target_sample_id"]))

print("Internal train:", len(internal_train), "positives:", int(internal_train["label_48h_final"].sum()),
      "regions:", internal_train["region_component_id"].nunique())
print("Internal val:", len(internal_val), "positives:", int(internal_val["label_48h_final"].sum()),
      "regions:", internal_val["region_component_id"].nunique())
print("Region overlap:", 0)

if internal_val["label_48h_final"].sum() == 0:
    raise RuntimeError("Internal validation has no positives; stop.")


## 3. Training-only robust channel normalisation

Fit preprocessing from internal-training files only:
- deterministic sample of unique training frames;
- per-channel robust absolute scale;
- `asinh(x / scale)`;
- per-channel z-score.


In [ ]:
LOCAL_COLS = ["local_tminus288","local_tminus192","local_tminus96"]

train_files = pd.unique(pd.concat([internal_train[c] for c in LOCAL_COLS], ignore_index=True))

rng = np.random.default_rng(SEED)
n_stats_files = min(1000, len(train_files))
stats_files = rng.choice(train_files, size=n_stats_files, replace=False)

PIXELS_PER_FILE = 1024
sample_chunks = [[] for _ in range(6)]

for i, path in enumerate(stats_files, 1):
    with np.load(path, allow_pickle=False) as z:
        x = z["x"]
        channels = [str(v) for v in z["channels"].tolist()]
        assert x.shape == (512,512,6)
        assert x.dtype == np.float32
        assert channels == EXPECTED_CHANNELS

        flat = x.reshape(-1, 6)
        take = rng.choice(flat.shape[0], size=min(PIXELS_PER_FILE, flat.shape[0]), replace=False)
        vals = flat[take]
        for c in range(6):
            sample_chunks[c].append(vals[:, c].astype(np.float64, copy=False))

    if i % 200 == 0:
        print(f"Stats files: {i}/{n_stats_files}", flush=True)

raw_samples = [np.concatenate(v) for v in sample_chunks]

channel_scale = []
for v in raw_samples:
    q = np.percentile(np.abs(v), 75)
    channel_scale.append(q if q > 1e-12 else 1.0)
channel_scale = np.asarray(channel_scale, dtype=np.float32)

transformed = [np.arcsinh(v / channel_scale[i]) for i, v in enumerate(raw_samples)]
channel_mean = np.asarray([v.mean() for v in transformed], dtype=np.float32)
channel_std = np.asarray([v.std() if v.std() > 1e-6 else 1.0 for v in transformed], dtype=np.float32)

normalisation = {
    "transform": "asinh(x / robust_training_scale), then z-score",
    "stats_files": int(n_stats_files),
    "pixels_per_file": int(PIXELS_PER_FILE),
    "channel_order": EXPECTED_CHANNELS,
    "channel_scale": channel_scale.tolist(),
    "channel_mean": channel_mean.tolist(),
    "channel_std": channel_std.tolist(),
    "fit_population": "internal_train_only",
}

(OUT / "normalisation.json").write_text(json.dumps(normalisation, indent=2) + "\n")
print(json.dumps(normalisation, indent=2))


## 4. Dataset and loader

In [ ]:
class TemporalAIADataset(Dataset):
    def __init__(self, frame, image_size=256):
        self.df = frame.reset_index(drop=True).copy()
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    def _load_frame(self, path):
        with np.load(path, allow_pickle=False) as z:
            x = z["x"]
            channels = [str(v) for v in z["channels"].tolist()]

            if x.shape != (512,512,6):
                raise ValueError(f"Unexpected x shape {x.shape}: {path}")
            if channels != EXPECTED_CHANNELS:
                raise ValueError(f"Channel mismatch {channels}: {path}")
            if not np.isfinite(x).all():
                raise ValueError(f"Nonfinite AIA tensor: {path}")

            x = x.astype(np.float32, copy=False)

        x = np.arcsinh(x / channel_scale.reshape(1,1,6))
        x = (x - channel_mean.reshape(1,1,6)) / channel_std.reshape(1,1,6)

        t = torch.from_numpy(x).permute(2,0,1).contiguous()

        if self.image_size != 512:
            t = F.interpolate(
                t.unsqueeze(0),
                size=(self.image_size, self.image_size),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)

        return t

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        seq = torch.stack([
            self._load_frame(r["local_tminus288"]),
            self._load_frame(r["local_tminus192"]),
            self._load_frame(r["local_tminus96"]),
        ], dim=0)

        y = torch.tensor(float(r["label_48h_final"]), dtype=torch.float32)
        return seq, y, r["target_sample_id"]

def make_loader(frame, shuffle, batch_size=BATCH_SIZE):
    return DataLoader(
        TemporalAIADataset(frame, IMAGE_SIZE),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        drop_last=False,
    )

train_loader = make_loader(internal_train, True)
val_loader = make_loader(internal_val, False)

xb, yb, sidb = next(iter(train_loader))
print("Batch x:", tuple(xb.shape))
print("Batch y:", tuple(yb.shape))
print("Example IDs:", list(sidb[:2]))
assert xb.shape[1:] == (3,6,IMAGE_SIZE,IMAGE_SIZE)


## 5. Shared CNN frame encoder + temporal GRU

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)

class FrameCNN(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(6, 32, 0.05),
            ConvBlock(32, 64, 0.05),
            ConvBlock(64, 128, 0.10),
            ConvBlock(128, 192, 0.10),
            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192, embed_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.proj(self.encoder(x))

class TemporalAIACNNGRU(nn.Module):
    def __init__(self, embed_dim=256, hidden_dim=192):
        super().__init__()
        self.frame_encoder = FrameCNN(embed_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        b,t,c,h,w = x.shape
        z = self.frame_encoder(x.reshape(b*t,c,h,w)).reshape(b,t,-1)
        _, hlast = self.gru(z)
        return self.head(hlast[-1]).squeeze(-1)

model = TemporalAIACNNGRU().to(DEVICE)

with torch.no_grad():
    test_logits = model(xb[:2].to(DEVICE, non_blocking=True))
print("Model output shape:", tuple(test_logits.shape))
print("Parameters:", sum(p.numel() for p in model.parameters()))


## 6. Threshold-free validation and training helpers

In [ ]:
def probability_metrics(y_true, p):
    y_true = np.asarray(y_true, dtype=int)
    p = np.asarray(p, dtype=float)
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
        "log_loss": float(log_loss(y_true, np.clip(p, 1e-7, 1-1e-7), labels=[0,1])),
    }

@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, ps, sids = [], [], []
    for x, y, sid in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        p = torch.sigmoid(logits).cpu().numpy()
        ys.extend(y.numpy().tolist())
        ps.extend(p.tolist())
        sids.extend(list(sid))
    return np.asarray(ys, dtype=int), np.asarray(ps, dtype=float), sids

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    losses = []
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


## 7. Development training — epoch selection on internal validation PR-AUC

In [ ]:
neg = int((internal_train["label_48h_final"] == 0).sum())
pos = int((internal_train["label_48h_final"] == 1).sum())
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))

best_pr = -np.inf
best_epoch = None
epochs_without_improvement = 0
history = []

best_path = OUT / "models" / "cnn_gru_internal_best.pt"

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)

    yv, pv, sv = predict(model, val_loader)
    m = probability_metrics(yv, pv)
    scheduler.step(m["pr_auc"])

    rec = {
        "epoch": epoch,
        "train_loss": loss,
        **m,
        "lr": float(optimizer.param_groups[0]["lr"]),
        "seconds": float(time.time() - t0),
    }
    history.append(rec)
    print(rec, flush=True)

    if m["pr_auc"] > best_pr + 1e-6:
        best_pr = m["pr_auc"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

pd.DataFrame(history).to_csv(OUT / "development_history.csv", index=False)

print("BEST_EPOCH:", best_epoch)
print("BEST_INTERNAL_VAL_PR_AUC:", best_pr)

if best_epoch is None:
    raise RuntimeError("No best epoch selected.")


## 8. Preserve internal-validation predictions

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
yv, pv, sv = predict(model, val_loader)

val_pred = pd.DataFrame({
    "target_sample_id": sv,
    "y_true": yv,
    "probability": pv,
})
val_pred.to_csv(
    OUT / "predictions" / "cnn_gru_internal_validation.csv.gz",
    index=False,
    compression="gzip",
)

internal_metrics = probability_metrics(yv, pv)
print(json.dumps(internal_metrics, indent=2))


## 9. Final Cycle-24 refit of the base model

Architecture and epoch count are frozen.
A new model is initialized from scratch and trained on the complete `cycle24_final_refit_pool`
for exactly `best_epoch` epochs. Calibration/threshold holdouts remain untouched.


In [ ]:
full_loader = make_loader(fit_pool, True)

final_model = TemporalAIACNNGRU().to(DEVICE)

neg_full = int((fit_pool["label_48h_final"] == 0).sum())
pos_full = int((fit_pool["label_48h_final"] == 1).sum())
full_pos_weight = torch.tensor([neg_full / pos_full], dtype=torch.float32, device=DEVICE)

final_criterion = nn.BCEWithLogitsLoss(pos_weight=full_pos_weight)
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
final_scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type=="cuda"))

refit_history = []

for epoch in range(1, best_epoch + 1):
    t0 = time.time()
    loss = train_one_epoch(final_model, full_loader, final_optimizer, final_criterion, final_scaler)
    rec = {
        "epoch": epoch,
        "train_loss": loss,
        "seconds": float(time.time() - t0),
    }
    refit_history.append(rec)
    print("REFIT", rec, flush=True)

pd.DataFrame(refit_history).to_csv(OUT / "final_refit_history.csv", index=False)

final_path = OUT / "models" / "cnn_gru_cycle24_final_refit.pt"
torch.save(final_model.state_dict(), final_path)
print("Saved:", final_path)


## 10. Save protocol and environment evidence

In [ ]:
protocol = {
    "status": "TEMPORAL_AIA_CNN_GRU_CYCLE24_BASE_MODEL_FROZEN_PENDING_CALIBRATION",
    "model": "shared_CNN_plus_GRU",
    "image_size_training": IMAGE_SIZE,
    "source_image_shape": [512,512,6],
    "channel_order": EXPECTED_CHANNELS,
    "history_lags_minutes": [-288,-192,-96],
    "forecast_horizon_hours": 48,
    "authoritative_label": "label_48h_final",
    "embedded_npz_y_used": False,
    "development_role": "cycle24_final_refit_pool",
    "calibration_role_used": False,
    "threshold_role_used": False,
    "cycle25_used": False,
    "epoch_selection_metric": "internal_validation_PR_AUC",
    "best_epoch": int(best_epoch),
    "best_internal_validation_metrics": internal_metrics,
    "loss": "BCEWithLogitsLoss with training-only positive class weight",
    "optimizer": "AdamW",
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "normalisation_file": str(OUT / "normalisation.json"),
    "final_model_path": str(final_path),
    "calibration_fitted": False,
    "threshold_selected": False,
    "scientific_clearance": False,
    "notes": [
        "Supervisor image-plus-temporal methodology implemented as CNN spatial encoder plus GRU temporal encoder.",
        "Cycle-25 remains sealed.",
        "Calibration and operating threshold are deferred to dedicated Cycle-24 holdouts.",
        "Earlier label-clearance and historical-source limitations remain inherited."
    ],
}

(OUT / "protocol_record.json").write_text(json.dumps(protocol, indent=2) + "\n")

env = {
    "python": sys.version,
    "torch": torch.__version__,
    "torch_cuda_runtime": torch.version.cuda,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
(OUT / "environment.json").write_text(json.dumps(env, indent=2) + "\n")

print(json.dumps(protocol, indent=2))
print(json.dumps(env, indent=2))


## 11. Handoff

19A3 will:
1. load this frozen Cycle-24 AIA base model;
2. fit probability calibration only on `cycle24_calibration_holdout`;
3. choose operating threshold only on `cycle24_threshold_holdout`;
4. freeze both before any Cycle-25 image evaluation.

No Cycle-25 score belongs in this notebook.
